
## 문제 정의
- **학습 데이터**: `train.csv` (generation=9 사람들의 completed 레이블 포함)
- **예측 대상**: `test.csv` (generation=10 사람들의 completed 예측)

## 1. Import & Data Loading

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, Pool
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings("ignore")

# 데이터 로드
train = pd.read_csv("train.csv", encoding="utf-8")
test = pd.read_csv("test.csv", encoding="utf-8")
submission = pd.read_csv("sample_submission.csv", encoding="utf-8")

print("데이터 로드 완료")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

데이터 로드 완료
Train shape: (748, 46)
Test shape: (814, 45)


## 2. 데이터 기본 정보 확인

In [2]:
print("\n" + "=" * 60)
print("Target 변수 분포")
print("=" * 60)
print(train['completed'].value_counts())
print(f"\n비율:{train['completed'].value_counts(normalize=True)}")


Target 변수 분포
completed
0    525
1    223
Name: count, dtype: int64

비율:completed
0    0.701872
1    0.298128
Name: proportion, dtype: float64


In [3]:
print("\n" + "=" * 60)
print("데이터 타입 분포")
print("=" * 60)
print(train.dtypes.value_counts())


데이터 타입 분포
str        34
float64     7
int64       4
bool        1
Name: count, dtype: int64


In [4]:
print("=" * 60)
print("결측치 분석 (Train)")
print("=" * 60)
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
if len(missing_train) > 0:
    print(f"\n결측치가 있는 컬럼 ({len(missing_train)}개):\n")
    for col, count in missing_train.items():
        pct = (count / len(train)) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")
else:
    print("\n결측치 없음")

print("\n" + "=" * 60)
print("결측치 분석 (Test)")
print("=" * 60)
missing_test = test.isnull().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)
if len(missing_test) > 0:
    print(f"\n결측치가 있는 컬럼 ({len(missing_test)}개):\n")
    for col, count in missing_test.items():
        pct = (count / len(test)) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")
else:
    print("\n결측치 없음")

결측치 분석 (Train)

결측치가 있는 컬럼 (18개):

  idea_contest: 748 (100.0%)
  contest_award: 748 (100.0%)
  class4: 747 (99.9%)
  contest_participation: 742 (99.2%)
  class3: 734 (98.1%)
  previous_class_5: 602 (80.5%)
  previous_class_8: 602 (80.5%)
  previous_class_7: 602 (80.5%)
  previous_class_6: 602 (80.5%)
  previous_class_4: 602 (80.5%)
  previous_class_3: 602 (80.5%)
  class2: 579 (77.4%)
  major1_2: 439 (58.7%)
  completed_semester: 28 (3.7%)
  major_field: 23 (3.1%)
  major type: 22 (2.9%)
  major1_1: 20 (2.7%)
  nationality: 1 (0.1%)

결측치 분석 (Test)

결측치가 있는 컬럼 (17개):

  idea_contest: 814 (100.0%)
  contest_participation: 814 (100.0%)
  contest_award: 814 (100.0%)
  class4: 807 (99.1%)
  class3: 768 (94.3%)
  previous_class_5: 659 (81.0%)
  previous_class_8: 659 (81.0%)
  previous_class_7: 659 (81.0%)
  previous_class_6: 659 (81.0%)
  previous_class_4: 659 (81.0%)
  previous_class_3: 659 (81.0%)
  class2: 572 (70.3%)
  major1_2: 183 (22.5%)
  major_field: 74 (9.1%)
  completed_semester:

In [5]:
print("=" * 60)
print("Train 데이터 샘플")
print("=" * 60)
train.head(3)

Train 데이터 샘플


,ID,generation,school1,major type,major1_1,major1_2,major_data,job,class1,class2,...,incumbents_company_level,incumbents_lecture_type,incumbents_lecture_scale,incumbents_lecture_scale_reason,interested_company,expected_domain,contest_participation,idea_contest,onedayclass_topic,completed
0,TRAIN_000,9,22,"복수 전공 ( 다중전공, 이중전공 포함 )",경제통상학,자연과학,False,대학생,1,4.0,...,해외 기업 (빅테크),"온, 오프라인 동시",100명 이상의 리스너와 10명 이상의 현직자,다양한 사람들과 만나서 생각을 교류할 수 있기 때문,"구글 딥마인드, 카카오 브레인","M. 전문, 과학 및 기술 서비스업",NaN,NaN,"Python 응용, 데이터 시각화 (Matplotlib, Seaborn 등), 머신...",0
1,TRAIN_001,9,1,"복수 전공 ( 다중전공, 이중전공 포함 )",자연과학,IT(컴퓨터 공학 포함),True,대학생,8,NaN,...,국내 빅테크 IT 계열 (네카라쿠배당토),오프라인,3~50명 내외의 강의 리스너와 1명의 현직자,더 많은 사람들이 있으면 제가 예상하지 못한 질문도 할 수 있다고 생각하기 때문입니다.,제일 기획,"J. 정보통신업, O. 공공 행정, 국방 및 사회보장 행정",NaN,NaN,머신러닝 / 딥러닝 응용,0
2,TRAIN_002,9,27,단일 전공,예체능,NaN,False,대학생,7,NaN,...,"국내 대기업 IT 계열 (금융, 제조 ...)",오프라인,3~50명 내외의 강의 리스너와 1명의 현직자,인원이 너무 적으면 서로 부담스러울 수 있을 것 같지만 너무 많으면 너무 피상적인 ...,Lg전자,"C. 제조업, K. 금융 및 보험업, R. 예술, 스포츠 및 여가관련 서비스업",NaN,NaN,"머신러닝 / 딥러닝 응용, SQL 응용, 웹 크롤링",0


## 3. 데이터 전처리

### 전처리 계획
1. 파생 변수 생성 (count_class, previous_class_count)
2. Bool 변환 (re_registration, contest_participation, major_type)
3. 칼럼 삭제 및 결측치 처리 (generation, contest_award, idea_contest, major type)
4. 다중값 처리(콤마->공백(단 괄호 안의 콤마는 언더바), 띄어쓰기 -> 언더바)

In [10]:
df = train.copy()
df_test = test.copy()

train['is_train'] = 1
test['is_train'] = 0

df_all = pd.concat([train, test], ignore_index=True)

# 1. 파생 변수 생성
## 1-1. count_class 칼럼 추가
class_cols = [f'class{i}' for i in [1,2,3,4]]
count_class_values = df_all[class_cols].notnull().sum(axis=1)
df_all.insert(df_all.columns.get_loc('class4')+1, 'count_class', count_class_values)
df_all = df_all.drop(columns=[c for c in ['major1_1','major1_2'] if c in df_all.columns])

## 1-2. previous_class_count 칼럼 추가
prev_class_cols = [f'previous_class_{i}' for i in range(3,9)]
prev_count_values = df_all[prev_class_cols].apply(
    lambda row: row.notnull() & (row != '해당없음'), axis=1
).sum(axis=1)
df_all.insert(df_all.columns.get_loc('previous_class_8')+1, 'previous_class_count', prev_count_values)

# 2. Boolean 변환
if 're_registration' in df_all.columns:
    df_all['re_registration'] = df_all['re_registration'].replace({'예': True, '아니요': False})
if 'contest_participation' in df_all.columns:
    df_all['contest_participation'] = df_all['contest_participation'].notnull()
if 'major type' in df_all.columns:
    df_all['major_type'] = df_all['major type'].replace({'복수 전공 ( 다중전공, 이중전공 포함 )': True, '단일 전공': False})

# 3. 칼럼 삭제 및 결측치 처리
cols_to_drop = ['generation', 'contest_award', 'idea_contest', 'major type']
df_all = df_all.drop(columns=[c for c in cols_to_drop if c in df_all.columns])
df_all = df_all.replace(r'^\s*$', np.nan, regex=True)

# 4. 다중값 처리 
## cols_to_space(다중값 칼럼): 공백->언더바, 콤마->공백
## cols_to_underbar(단일값 칼럼): 공백 -> 언더바
## cols_to_delete(단일 값에 콤마 존재): 콤마->삭제
## cols_in_paren(괄호 안에 콤마 존재): 콤마->언더바 incumbents_lecture_type
cols_to_space = ['major_field', 'desired_job', 'certificate_acquisition', 
                'desired_certificate', 'desired_job_except_data', 'interested_company', 'onedayclass_topic'] 
cols_to_underbar = ['inflow_route', 'whyBDA', 'what_to_gain', 'hope_for_group', 'previous_class_3', 'previous_class_4',
                   'previous_class_5', 'previous_class_6','previous_class_7','previous_class_8', 'major_field',
                   'desired_career_path', 'desired_job', 'certificate_acquisition', 'desired_certificate', 'desired_job_except_data', 'incumbents_level',
                   'incumbents_lecture', 'incumbents_company_level', 'incumbents_lecture_type', 'incumbents_lecture_scale_reason', 'interested_company', 'expected_domain', 'onedayclass_topic'] 
cols_to_delete = ['whyBDA']
cols_in_paren = ['major type', 'incumbents_company_level', 'onedayclass_topic']

for col in df_all.columns:
    if col in cols_to_delete:
        df_all[col] = df_all[col].str.replace(',', '')
    if col in cols_in_paren:
        df_all[col] = df_all[col].str.replace(r'\(([^)]*)\)', lambda m: '(' + m.group(1).replace(',', '_') + ')', regex=True)
    if col in cols_to_underbar:
        df_all[col] = df_all[col].str.replace(' ', '_')
        if col == 'incumbents_lecture_type':
            df_all[col] = df_all[col].str.replace(',_', '_')
    if col in cols_to_space:
        df_all[col] = df_all[col].str.replace(' ', '_')
        df_all[col] = df_all[col].str.replace(',_', ' ')

prev_class_cols = [f'previous_class_{i}' for i in range(3, 9)]
for col in prev_class_cols:
    if col in df_all.columns:
        df_all[col] = df_all[col].str.extract(r'^(\d{4})')[0]
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce').astype('Int64')

# Split back to train and test
df_train = df_all[df_all['is_train'] == 1].drop(columns=['is_train'])
df_test = df_all[df_all['is_train'] == 0].drop(columns=['is_train', 'completed'])

train_ids = df_train['ID']
test_ids = df_test['ID']
y_target = df_train['completed']
df_train = df_train.drop(['ID', 'completed'], axis=1)
df_test = df_test.drop(['ID'], axis=1)
df_train.to_csv('train_v8.csv', index=False)
df_test.to_csv('test_v8.csv', index=False)

print(f"전처리 완료! Train: {df.shape} -> {df_train.shape}, Test: {test.shape} -> {df_test.shape}")
print("전처리 완료된 데이터를 저장했습니다.")

전처리 완료! Train: (748, 46) -> (748, 41), Test: (814, 46) -> (814, 41)
전처리 완료된 데이터를 저장했습니다.
